In [1]:
import os
import torch

from tqdm import tqdm

from stock_gpt import StockGPT, StockLinearModel
from dataloader_builder import build_dataloaders
from setup import Stock_GPT_cfg, Linear_Model_cfg
from setup import path_data_preprocessor

In [2]:
cuda = True if torch.cuda.is_available() else False

print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PyTorch: 2.13.0+cu132
CUDA build: 13.2
CUDA available: True
GPU: NVIDIA GeForce RTX 4070 Laptop GPU


In [3]:
def batch_loss(x, y, model, fn):
    """
    Assume x and y are on the correct device already
    """
    x = model(x)
    y_norm = (y - model.target_mean) / model.target_std
    return fn(x, y_norm)

def loader_loss(data_loader, model, device, fns: dict = {}, max_batches = float("inf"), pbar = None, desc=""):
    """
    Returns a dict of loss calcualted using all loss functions in fns
    """
    num_batches = min(len(data_loader), max_batches)
    avg_metrics = {f: 0 for f in fns}
    for i, (p, t) in enumerate(data_loader):
        p = p.to(device, non_blocking=True)
        t = t.to(device, non_blocking=True)
        if i == num_batches:
            break
        for name, fn in fns.items():
            avg_metrics[name] += (batch_loss(p, t, model, fn) - avg_metrics[name])/(i+1)

        if pbar is not None:
            pbar.update(1)
            if i % max(1,int(num_batches*0.001))==0:
                pbar.set_description(f"{desc} ({i}/{num_batches}) [{pbar.n}/{pbar.total}]")
    return avg_metrics

## MODEL TRAINING ---------------------------

In [4]:
def train_model_cuda(model, device, optimizer, cuda_scaler, max_epochs,
                     train_dl, val_dl, train_fn, eval_fns, eval_bs):
    #* LOADS MODEL
    if os.path.exists(model.checkpoint_path):
        checkpoint = load_model(model.checkpoint_path, model, device, optimizer, cuda_scaler)
        bvm, epoch, train_losses, val_losses = (
            checkpoint["bvm"], checkpoint["epoch"]+1, checkpoint["train_losses"], checkpoint["val_losses"]
        )
    else:
        bvm, epoch, train_losses, val_losses = float("inf"), 0, [], []

    eval_steps = min(eval_bs, len(train_dl)) + min(eval_bs, len(val_dl))
    pbar = tqdm(total=(max_epochs-epoch)*(len(train_dl)+eval_steps), desc=f"Setting up...".ljust(80),
                bar_format="|{bar}| {percentage:3.1f}% ({elapsed}) {desc}", position=0, leave=False)
    try:
        for epoch in range(epoch, max_epochs):
            #* TRAINS MODEL
            model.train() 
            for x, y in train_dl:
                x = x.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)

                with torch.autocast(device_type="cuda",dtype=torch.float16):
                    loss = batch_loss(x, y, model, train_fn)
                cuda_scaler.scale(loss).backward()
                cuda_scaler.step(optimizer)
                cuda_scaler.update()

                pbar.update(1)
                if (pbar.n % max(1,int(pbar.total*0.001))==0):
                    pbar.set_description(f"Training the model... [{pbar.n}/{pbar.total}]")

            #* EVALUATES MODEL
            model.eval()
            pbar.set_description(f"Evaluating Epoch {epoch}... [{pbar.n}/{pbar.total}]")
            train_metrics, val_metrics = evaluate_model(train_dl, val_dl, model, device, eval_fns, eval_bs)
            pbar.write((f"\n{'-'*100}\n"
                        f"Epoch {epoch+1}:"
                        f"Training Loss = {train_metrics['MAE Loss'].mean()}\n"
                        f"Validation Loss = {val_metrics['MAE Loss'].mean()}"))
            train_losses.append(train_metrics)
            val_losses.append(val_metrics)

            #* SAVES MODEL
            cvm = val_metrics['MAE Loss'].mean().item()
            checkpoint = {
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "cuda_scaler": cuda_scaler.state_dict(),
                "epoch": epoch,
                "train_losses": train_losses,
                "val_losses": val_losses,
                "bvm": bvm
            }
            if (cvm < bvm):
                bvm = cvm
                checkpoint["bvm"] = cvm
                torch.save(checkpoint, model.best_path)
                pbar.write((f"Best Validation MEA: {bvm}"))
            torch.save(checkpoint, model.checkpoint_path)
    finally:
            pbar.close()
    return train_losses, val_losses

In [5]:
def load_model(path, model, device, optimizer=None, cuda_scaler=None):
    checkpoint = torch.load(path, map_location=device)
    model.load_state_dict(checkpoint["model"])
    if optimizer is not None and "optimizer" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer"])
    if cuda_scaler is not None and "cuda_scaler" in checkpoint:
        cuda_scaler.load_state_dict(checkpoint["cuda_scaler"])
    return checkpoint

def evaluate_model(train_dl, val_dl, model, device, eval_fns, eval_bs, pbar = None):
    """
    Returns a list of dictionaries, with each dictionary correspoding to a function in eval_fns
    """
    with torch.no_grad():
        train_metrics = loader_loss(train_dl, model, device, eval_fns, eval_bs, pbar,
                                    desc="Evaluating model on training data...")
        val_metrics = loader_loss(val_dl, model, device, eval_fns, eval_bs, pbar,
                                  desc="Evaluating model on validation data...")    
    return train_metrics, val_metrics

def evaluate_best_model(model, device, optimizer, cuda_scaler, train_dl, val_dl, eval_fns, eval_bs):
    load_model(model, device, optimizer, cuda_scaler)
    return evaluate_model(train_dl, val_dl, model, device, eval_fns, eval_bs)


In [6]:
torch.manual_seed(1234)
train_dl, val_dl, test_dl, train_norms = build_dataloaders(path_data_preprocessor)

Reading source path at preprocessed_data/data_15min_2025.parquet...
Building DataLoaders...


In [7]:
stockGPT = StockGPT(Stock_GPT_cfg, train_norms)
model_params = sum(p.numel() for p in stockGPT.parameters())
print(model_params)
stockGPT.to(device)

stockLinear = StockLinearModel(Linear_Model_cfg, train_norms)
linear_params = sum(p.numel() for p in stockLinear.parameters())
print(linear_params)
stockLinear.to(device)

optimizer1 = torch.optim.AdamW(stockGPT.parameters(), lr=0.0004, weight_decay=0.1)
optimizer2 = torch.optim.AdamW(stockLinear.parameters(), lr=0.0004, weight_decay=0.1)

train_fn = torch.nn.HuberLoss()
eval_fns = {
    "Huber Loss": torch.nn.HuberLoss(reduction="none"),
    "MAE Loss": torch.nn.L1Loss(reduction="none")
}

max_epochs = 10
eval_bs = 1000

scaler1 = torch.amp.GradScaler("cuda")
scaler2 = torch.amp.GradScaler("cuda")

naive_train_losses, naive_val_losses = train_model_cuda(stockLinear, device, optimizer2, scaler2, max_epochs,
                                                        train_dl, val_dl, train_fn, eval_fns, eval_bs)
model_train_losses, model_val_losses = train_model_cuda(stockGPT, device, optimizer1, scaler1, max_epochs, 
                                                        train_dl, val_dl, train_fn, eval_fns, eval_bs)

6324992
6144


## Model Analysis -------------------------

In [8]:
def process_losses(losses: list[dict], key = "MAE Loss"):
    return [loss_dict[key](dim=(0,1)) for loss_dict in losses]

def tensor_to_string(t, cs):
    return "".join(f"{v.item():<{cs}.4f}" for v in t)

def format_num(n):
    if n >= 1e9:
        return f"{n / 1e9:.1f}B"
    if n >= 1e6:
        return f"{n / 1e6:.1f}M"
    if n >= 1e3:
        return f"{n / 1e3:.1f}K"
    return str(n)

def print_losses(losses, model_names, parameters, col_names, cs = 9):
    title = f"MAE Loss\n"
    bound = f"\n{'-'*110}\n\n"
    header1 = f"{' '*20}"+"".join(f"{col_name:<{cs}}" for col_name in col_names)+"\n"
    rows = "".join(
        f"{row_name}: {parameters[i]}\n"
        f"    Training:       {tensor_to_string(losses[i*2], cs)}  >  {losses[i*2].mean():.4f}\n"
        f"    Validation:     {tensor_to_string(losses[i*2+1], cs)}  >  {losses[i*2+1].mean():.4f}\n"
        f"    "
        f"\n"
    for i, row_name in enumerate(model_names))
    output = [
        bound,
        title,
        bound,
        header1,
        rows,
        bound
    ]
    print("".join(output))

In [ ]:
eval_steps = min(eval_bs, len(train_dl)) + min(eval_bs, len(val_dl))
eval_pbar = tqdm(total=2*eval_steps, desc=f"Evaluating the best model parameters...".ljust(80),
                bar_format="|{bar}| {percentage:3.1f}% ({elapsed}) {desc}", position=0, leave=False)
gpt_losses = evaluate_model(train_dl, val_dl, stockGPT, device, eval_fns, eval_bs, eval_pbar)
linear_losses = evaluate_model(train_dl, val_dl, stockLinear, device, eval_fns, eval_bs, eval_pbar)

print_losses(process_losses(gpt_losses + linear_losses, 
                            "MAE Loss"),
                            ["StockGPT", "LinearModel"],
                            [f"{format_num(model_params)}", f"{format_num(linear_params)}"],
                            Stock_GPT_cfg["target_features"])

|          | 0.0% (00:00) Evaluating the best model parameters...                                         

In [ ]:
mtld, mvld, ntld, nvld = model_train_losses, model_val_losses, 
## lists of dictionaries keys (MAE Loss and Huber Loss) of tensors shape (256, 25, 10)
## only need to print last / best

print(len(mtld))
mtml = mtld[-1]["MAE Loss"]
print(mtml.shape)
mtmlpc = mtml.mean(dim=(0,1))
print(mtmlpc)
print(mtmlpc.mean())

print(Stock_GPT_cfg["target_features"])

ValueError: not enough values to unpack (expected 4, got 2)

In [ ]:
print(len(train_dl))
pbar = tqdm(train_dl, total=len(train_dl), desc=f"[{0}/{len(train_dl)}] Entering training data...".ljust(80),
            bar_format="|{bar}| {percentage:3.1f}% ({elapsed}) {desc}")

res = ""
for input_batch, target_batch in iter(train_dl):
    sample = model(input_batch)
    pbar.update(1)
    pbar.set_description_str((f"[{pbar.n}/{pbar.total}] "
                              f"Entering training data...".ljust(80)))
    res = sample
    break
print(res)
print(res.shape)

test_predict = torch.randn(1, 10, 12)
out = model(test_predict)
out = out[0][-1]
print(out)
print(out.shape)